In [ ]:
"""
Mixed xLSTM model for ECG5000 time-series classification.

Architecture:
Block 0 = mLSTM
Block 1 = sLSTM
"""

# Install packages not already provided by Colab
!pip install -q git+https://github.com/NX-AI/xlstm.git

from google.colab import drive
drive.mount("/content/drive")

import time
from typing import Tuple

import numpy as np
import torch
import torch.nn as nn
from matplotlib import pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from xlstm import (
    xLSTMBlockStack,
    xLSTMBlockStackConfig,
    mLSTMBlockConfig,
    mLSTMLayerConfig,
    sLSTMBlockConfig,
    sLSTMLayerConfig,
)
DATA_DIR = "/content/drive/MyDrive/TUBerlin_Data"

TRAIN_PATH = f"{DATA_DIR}/ECG5000_FOLD_1_TRAIN_3200.txt"
VAL_PATH = f"{DATA_DIR}/ECG5000_FOLD_1_VAL_800.txt"
TEST_PATH = f"{DATA_DIR}/ECG5000_FINAL_TEST_1000.txt"


class XLSTMClassifier(nn.Module):
    def __init__(
        self,
        input_size: int,
        d_model: int,
        n_classes: int,
        num_blocks: int,
        num_heads: int,
        context_length: int,
        conv1d_kernel_size: int,
        qkv_proj_blocksize: int,
    ):
        super().__init__()

        self.input_projection = nn.Linear(input_size, d_model)

        xlstm_config = xLSTMBlockStackConfig(
            mlstm_block=mLSTMBlockConfig(
                mlstm=mLSTMLayerConfig(
                    conv1d_kernel_size=conv1d_kernel_size,
                    qkv_proj_blocksize=qkv_proj_blocksize,
                    num_heads=num_heads,
                )
            ),
            slstm_block=sLSTMBlockConfig(
                slstm=sLSTMLayerConfig(
                    num_heads=num_heads,
                    conv1d_kernel_size=conv1d_kernel_size,
                    backend="vanilla",
                )
            ),
            context_length=context_length,
            num_blocks=num_blocks,
            embedding_dim=d_model,
            slstm_at=[1],
        )

        self.encoder = xLSTMBlockStack(xlstm_config)

        self.output_projection = nn.Linear(
            d_model * context_length,
            n_classes,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_projection(x)
        z = self.encoder(x)
        z = z.flatten(start_dim=1)
        output = self.output_projection(z)

        return output


def load_ecg5000_txt(
    file_path: str,
    delimiter=None,
) -> Tuple[np.ndarray, np.ndarray]:

    raw = np.loadtxt(file_path, delimiter=delimiter)

    labels = raw[:, 0].astype(np.int64)
    labels = labels - 1

    data = raw[:, 1:].astype(np.float32)
    data = data[..., np.newaxis]

    return data, labels


class ECG5000Dataset(Dataset):
    def __init__(self, data: np.ndarray, labels: np.ndarray):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


def plot_sample(
    data: np.ndarray,
    labels: np.ndarray,
    sample_idx: int,
):
    sample = data[sample_idx, :, 0]
    label = labels[sample_idx] + 1
    time_steps = np.arange(sample.shape[0])

    plt.figure(figsize=(8, 3))
    plt.plot(time_steps, sample, linewidth=1.5)
    plt.title(f"ECG5000 sample — Class: {label}")
    plt.xlabel("Time step")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(
    true_indices,
    predicted_indices,
):
    cm = confusion_matrix(
        true_indices,
        predicted_indices,
    )

    class_names = ["1", "2", "3", "4", "5"]

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    )

    fig, ax = plt.subplots(figsize=(6, 5))

    display.plot(
        ax=ax,
        cmap="Blues",
        values_format="d",
    )

    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title("Mixed xLSTM Confusion Matrix – ECG5000")
    plt.tight_layout()
    plt.show()


def main():
    torch.manual_seed(42)
    np.random.seed(42)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Device:", device)

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

    # Load predefined stratified fold
    train_data, train_labels = load_ecg5000_txt(TRAIN_PATH)
    val_data, val_labels = load_ecg5000_txt(VAL_PATH)
    test_data, test_labels = load_ecg5000_txt(TEST_PATH)

    print("\nDATA")
    print("Train shape:", train_data.shape)
    print("Val shape:", val_data.shape)
    print("Test shape:", test_data.shape)
    print("Classes:", np.unique(train_labels) + 1)

    plot_sample(train_data, train_labels, sample_idx=0)

    # Scale signals using training data only
    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_data.reshape(-1, train_data.shape[-1])
    ).reshape(train_data.shape)

    val_scaled = scaler.transform(
        val_data.reshape(-1, val_data.shape[-1])
    ).reshape(val_data.shape)

    test_scaled = scaler.transform(
        test_data.reshape(-1, test_data.shape[-1])
    ).reshape(test_data.shape)

    train_dataset = ECG5000Dataset(train_scaled, train_labels)
    val_dataset = ECG5000Dataset(val_scaled, val_labels)
    test_dataset = ECG5000Dataset(test_scaled, test_labels)

    # Hyperparameters
    batch_size = 32
    learning_rate = 0.0001
    d_model = 64
    num_blocks = 2
    num_heads = 4
    conv1d_kernel_size = 4
    qkv_proj_blocksize = 4
    epochs = 100

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    input_size = train_data.shape[-1]
    n_timesteps = train_data.shape[1]
    n_classes = len(np.unique(train_labels))

    model = XLSTMClassifier(
        input_size=input_size,
        d_model=d_model,
        n_classes=n_classes,
        num_blocks=num_blocks,
        num_heads=num_heads,
        context_length=n_timesteps,
        conv1d_kernel_size=conv1d_kernel_size,
        qkv_proj_blocksize=qkv_proj_blocksize,
    ).to(device)

    num_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    print(f"\nModel has {num_params:,} trainable parameters.")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    criterion = nn.CrossEntropyLoss()

    train_losses = []
    val_losses = []

    best_val_loss = float("inf")
    best_val_epoch = 0
    time_per_epoch = []

    checkpoint_path = "best_model_xlstm.pth"

    for epoch in range(epochs):
        start_time = time.time()

        model.train()
        train_loss = 0.0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            predictions = model(x_batch)

            loss = criterion(
                predictions,
                y_batch,
            )

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        model.eval()

        val_loss = 0.0
        val_predictions = []
        val_targets = []

        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                predictions = model(x_batch)

                loss = criterion(
                    predictions,
                    y_batch,
                )

                val_loss += loss.item()

                predicted_classes = predictions.argmax(dim=1)

                val_predictions.extend(
                    predicted_classes.cpu().numpy()
                )

                val_targets.extend(
                    y_batch.cpu().numpy()
                )

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        val_f1 = f1_score(
            val_targets,
            val_predictions,
            average="macro",
            zero_division=0,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_epoch = epoch + 1

            torch.save(
                model.state_dict(),
                checkpoint_path,
            )

        epoch_time = time.time() - start_time
        time_per_epoch.append(epoch_time)

        print(
            f"Epoch {epoch + 1:3d}/{epochs} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"Val Macro-F1: {val_f1:.4f} | "
            f"Best Epoch: {best_val_epoch} | "
            f"Time: {epoch_time:.2f}s"
        )

    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Mixed xLSTM Training")
    plt.yscale("log")
    plt.grid(linestyle="dashed")
    plt.legend()
    plt.tight_layout()
    plt.show()

    best_model = XLSTMClassifier(
        input_size=input_size,
        d_model=d_model,
        n_classes=n_classes,
        num_blocks=num_blocks,
        num_heads=num_heads,
        context_length=n_timesteps,
        conv1d_kernel_size=conv1d_kernel_size,
        qkv_proj_blocksize=qkv_proj_blocksize,
    ).to(device)

    best_model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device,
        )
    )

    best_model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.to(device)

            logits = best_model(x_batch)
            predicted_classes = logits.argmax(dim=1)

            all_predictions.extend(
                predicted_classes.cpu().numpy()
            )

            all_targets.extend(
                y_batch.numpy()
            )

    all_predictions = np.array(all_predictions)
    all_targets = np.array(all_targets)

    accuracy = accuracy_score(
        all_targets,
        all_predictions,
    )

    precision = precision_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0,
    )

    recall = recall_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0,
    )

    f1_macro = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0,
    )

    average_time_per_epoch = (
        sum(time_per_epoch) / len(time_per_epoch)
    )

    print("\nFINAL MIXED xLSTM RESULTS")
    print(f"Best validation epoch: {best_val_epoch}")
    print(f"Best validation loss: {best_val_loss:.6f}")
    print(f"Test Accuracy: {accuracy * 100:.2f}%")
    print(f"Test Precision (Macro): {precision:.4f}")
    print(f"Test Recall (Macro): {recall:.4f}")
    print(f"Test F1 (Macro): {f1_macro:.4f}")
    print(f"Average time per epoch: {average_time_per_epoch:.2f}s")

    plot_confusion_matrix(
        all_targets,
        all_predictions,
    )


if __name__ == "__main__":
    main()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 122.1 MB/s eta 0:00:0000:01
CUDA available: True
CUDA version: 12.8
Device: cuda
GPU: Tesla T4


FileNotFoundError: /content/drive/MyDrive/TUBerlin_Data/ECG5000_FOLD_1_TRAIN_3200.txt not found.